In [ ]:
import os
import django

# Replace 'myproject' with your actual Django project directory name
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "datamodel_demo.settings")

# Prevents synchronous operation errors in the asynchronous Jupyter loop
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()

In [2]:
from app1.models import Entity, EntityType
from app1.utils import (
    descendant_types, entries_of_type, group_by_type, count_by_type,
    traverse_entities, latest_information_record,
)

In [9]:
[str(eo.parent)+'/'+eo.code for eo in EntityType.objects.all()]

['None/material_entity',
 'None/biological_source',
 'biological_source/brain_tissue',
 'brain_tissue/fresh_frozen_brain',
 'brain_tissue/tissue_section',
 'tissue_section/stained_section',
 'stained_section/spatial_library',
 'None/digital_entity',
 'digital_entity/fastq_file',
 'digital_entity/expression_matrix',
 'digital_entity/spatial_position_file',
 'digital_entity/cluster_result',
 'digital_entity/analysis_report']

In [4]:
from django.db.models import Count
from IPython.display import display
from app1.models import (
    Entity, EntityInformationRecord, EntityType, EntityTypeRecordSlot,
    InformationRecordType,
)

print("Entity types:", EntityType.objects.count())
print("Information-record types:", InformationRecordType.objects.count())
print("Entities:", Entity.objects.count())
print("Entity information records:", EntityInformationRecord.objects.count())

print("\nEntityType overview")
display(list(
    EntityType.objects.annotate(entity_count=Count("entities"))
    .order_by("code")
    .values("code", "name", "is_instantiable", "entity_count")
))

print("\nInformationRecordType overview")
display(list(
    InformationRecordType.objects.annotate(record_count=Count("entityinformationrecord"))
    .order_by("code")
    .values("code", "name", "is_instantiable", "record_count")
))

Entity types: 13
Information-record types: 0
Entities: 18
Entity information records: 18

EntityType overview


[{'code': 'analysis_report',
  'name': 'Analysis report',
  'is_instantiable': True,
  'entity_count': 1},
 {'code': 'biological_source',
  'name': 'Biological source',
  'is_instantiable': False,
  'entity_count': 0},
 {'code': 'brain_tissue',
  'name': 'Brain tissue',
  'is_instantiable': True,
  'entity_count': 1},
 {'code': 'cluster_result',
  'name': 'Cluster result',
  'is_instantiable': True,
  'entity_count': 1},
 {'code': 'digital_entity',
  'name': 'Digital entity',
  'is_instantiable': False,
  'entity_count': 0},
 {'code': 'expression_matrix',
  'name': 'Expression matrix',
  'is_instantiable': True,
  'entity_count': 1},
 {'code': 'fastq_file',
  'name': 'FASTQ sequencing file',
  'is_instantiable': True,
  'entity_count': 2},
 {'code': 'fresh_frozen_brain',
  'name': 'Fresh frozen brain',
  'is_instantiable': True,
  'entity_count': 1},
 {'code': 'material_entity',
  'name': 'Material entity',
  'is_instantiable': True,
  'entity_count': 7},
 {'code': 'spatial_library',
 


InformationRecordType overview


[]

## Type hierarchies

Inspect the roots and descendants in each tier-2 vocabulary. The shared utility handles descendant traversal safely.

In [5]:
def hierarchy_rows(model):
    rows = []
    for root in model.objects.filter(parent__isnull=True).order_by("code"):
        for node in descendant_types(root):
            depth = 0
            parent = node.parent
            while parent is not None:
                depth += 1
                parent = parent.parent
            rows.append({
                "code": node.code,
                "name": node.name,
                "depth": depth,
                "parent": node.parent.code if node.parent else None,
                "is_instantiable": node.is_instantiable,
            })
    return rows

print("EntityType hierarchy")
display(hierarchy_rows(EntityType))

print("InformationRecordType hierarchy")
display(hierarchy_rows(InformationRecordType))

EntityType hierarchy


[{'code': 'biological_source',
  'name': 'Biological source',
  'depth': 0,
  'parent': None,
  'is_instantiable': False},
 {'code': 'brain_tissue',
  'name': 'Brain tissue',
  'depth': 1,
  'parent': 'biological_source',
  'is_instantiable': True},
 {'code': 'fresh_frozen_brain',
  'name': 'Fresh frozen brain',
  'depth': 2,
  'parent': 'brain_tissue',
  'is_instantiable': True},
 {'code': 'spatial_library',
  'name': 'Spatial transcriptomics library',
  'depth': 4,
  'parent': 'stained_section',
  'is_instantiable': True},
 {'code': 'stained_section',
  'name': 'Stained tissue section',
  'depth': 3,
  'parent': 'tissue_section',
  'is_instantiable': True},
 {'code': 'tissue_section',
  'name': 'Tissue section',
  'depth': 2,
  'parent': 'brain_tissue',
  'is_instantiable': True},
 {'code': 'analysis_report',
  'name': 'Analysis report',
  'depth': 1,
  'parent': 'digital_entity',
  'is_instantiable': True},
 {'code': 'cluster_result',
  'name': 'Cluster result',
  'depth': 1,
  'par

InformationRecordType hierarchy


[]

## Record coverage and constraints

Compare current information-record usage with the type vocabulary and inspect the entity-type rules that describe expected sidecars.

In [6]:
record_usage = list(
    EntityInformationRecord.objects
    .values("information_record_type__code")
    .annotate(record_count=Count("pk"), entity_count=Count("entity", distinct=True))
    .order_by("information_record_type__code")
)

print("Entity information-record usage")
display(record_usage)

print("Records without a type:", EntityInformationRecord.objects.filter(
    information_record_type__isnull=True
).count())

print("EntityTypeRecordSlot rules")
display(list(
    EntityTypeRecordSlot.objects.select_related("entity_type", "record_type")
    .order_by("entity_type__code", "record_type__code")
    .values(
        "entity_type__code", "record_type__code",
        "min_count", "max_count", "match_mode",
    )
))

Entity information-record usage


[{'information_record_type__code': None,
  'record_count': 18,
  'entity_count': 18}]

Records without a type: 18
EntityTypeRecordSlot rules


[]